# Ella2027 — LTX-2.5 22B Distilled on Kaggle (Run All)

دفتر الاستعادة الرسمي لمزوّد Ella2027. كل خطوات الإعداد موجودة هنا: فحص GPU والإنترنت، تثبيت WanGP مع الحفاظ على PyTorch الخاص بـ Kaggle، تنزيل ملفات LTX-2.5 المطلوبة أو إعادة استخدام الكاش، تسجيل ملف finetune المحلي، تنزيل عامل Ella2027 من هذا الفرع من GitHub، ثم تشغيله بالخلفية.

**قبل Run All مرة واحدة:** فعّل GPU وInternet من إعدادات Kaggle، وأضف Kaggle Secret باسم `ELLA_KAGGLE_WORKER_TOKEN` بنفس قيمة `KAGGLE_LTX_WORKER_TOKEN` في Render. لا تضع التوكن في الدفتر أو GitHub.

يستخدم هذا الدفتر WanGP v20 عبر `shared.api` والنموذج `ltx2_25_22B_distilled`. أول تشغيل في Runtime جديد ينزّل الأوزان الكبيرة. تبقى حدود جلسات Kaggle والذاكرة وسرعة GPU قائمة.

## 1. Preflight — GPU, internet, and storage
Run this first. It does not download model weights.

In [ ]:
import os, shutil, torch, subprocess, sys
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name} | VRAM {p.total_memory/1024**3:.2f} GB')
MODEL_DIR = '/tmp/ella-ltx25'
os.makedirs(MODEL_DIR, exist_ok=True)
for path in ['/tmp', '/kaggle/working']:
    u = shutil.disk_usage(path)
    print(f'{path}: free={u.free/1024**3:.1f} GB | total={u.total/1024**3:.1f} GB')
print('Model directory:', MODEL_DIR)
print('Preflight complete.')

## 2. Get WanGP source
The repository is cloned into the small working area. Model weights are stored separately in `/tmp`.

In [ ]:
import os, subprocess
WGP_DIR = '/kaggle/working/WanGP'
if not os.path.isdir(os.path.join(WGP_DIR, '.git')):
    if os.path.exists(WGP_DIR):
        raise RuntimeError(f'{WGP_DIR} exists but is not a Git checkout. Rename/remove it before continuing.')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/deepbeepmeep/WanGP.git', WGP_DIR], check=True)
else:
    print('WanGP checkout already exists; keeping it unchanged.')
os.chdir(WGP_DIR)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
if not os.path.isfile(os.path.join(WGP_DIR, 'shared', 'api.py')):
    raise RuntimeError('This checkout does not include shared/api.py; the current WanGP API source is required.')
print('WanGP source and Python API ready:', WGP_DIR)


## 3. Install dependencies without replacing Kaggle's CUDA PyTorch
Pin the already-installed Torch family during requirements installation. This avoids an unnecessary large Torch/CUDA replacement.

In [ ]:
import os, sys, subprocess, torch
import importlib.util
from pathlib import Path
os.chdir(WGP_DIR)
constraints = Path('/tmp/ella-kaggle-constraints.txt')
pins = [f'torch=={torch.__version__.split("+")[0]}']
for package in ['torchvision', 'torchaudio']:
    if importlib.util.find_spec(package) is not None:
        mod = __import__(package)
        pins.append(f'{package}=={mod.__version__.split("+")[0]}')
constraints.write_text('\n'.join(pins) + '\n')
print('Constraints:', constraints.read_text())
if not Path('requirements.txt').is_file():
    raise FileNotFoundError('WanGP requirements.txt was not found.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-c', str(constraints), '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'huggingface_hub>=0.30.0'], check=True)
print('Dependency installation complete. If Kaggle asks for a runtime restart, restart once and rerun cells from section 1.')

## 4. Download the two required LTX-2.5 checkpoints
Downloads are resumable through Hugging Face Hub's cache. If a download is interrupted, rerun this cell; completed files are reused. Do not save these large files in `/kaggle/working`.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download
MODEL_DIR = '/tmp/ella-ltx25'
os.makedirs(MODEL_DIR, exist_ok=True)
main_name = 'ltx-2.5-22b-distilled_diffusion_model_nvfp4.safetensors'
encoder_name = 'gemma4-12b-ltx-v1_int8_convrot.safetensors'
print('Downloading main LTX-2.5 checkpoint (large file; resumable)...')
main_path = hf_hub_download(repo_id='DeepBeepMeep/LTX-2', filename=main_name, local_dir=MODEL_DIR)
print('Main checkpoint ready:', main_path)
print('Downloading Gemma 4 12B INT8 ConvRot text encoder...')
encoder_path = hf_hub_download(repo_id='DeepBeepMeep/LTX-2', filename='gemma4-12b-ltx-v1/' + encoder_name, local_dir=MODEL_DIR)
print('Text encoder ready:', encoder_path)
for p in [Path(main_path), Path(encoder_path)]:
    if not p.is_file() or p.stat().st_size < 100_000_000:
        raise RuntimeError(f'Checkpoint missing or unexpectedly small: {p}')
    print(f'VERIFIED {p.name}: {p.stat().st_size/1024**3:.2f} GB')
print('\nBoth required checkpoints are present.')

# Register the downloaded NVFP4 transformer and Gemma encoder as a local LTX-2.5 finetune.
# This is necessary because a checkpoint file in /tmp is not automatically selected by WanGP.
import json
finetunes_dir = Path(WGP_DIR) / "finetunes"
finetunes_dir.mkdir(parents=True, exist_ok=True)
finetune_profile = {
    "model": {
        "name": "LTX-2.5 22B Distilled NVFP4 — Ella local",
        "architecture": "ltx2_25_22B",
        "description": "Ella2027 local LTX-2.5 NVFP4 checkpoint and Gemma 4 INT8 ConvRot encoder.",
        "URLs": [str(Path(main_path).resolve())],
        "text_encoder_URLs": [str(Path(encoder_path).resolve())],
        "ltx2_pipeline": "distilled"
    }
}
finetune_path = finetunes_dir / "ltx2_25_22B_distilled.json"
finetune_path.write_text(json.dumps(finetune_profile, indent=2), encoding="utf-8")
print("Local LTX-2.5 finetune registered:", finetune_path)
print(finetune_path.read_text())


## 5. Start ELLA Gradio Studio (connected to Render)

This cell initializes WanGP once, exposes the real LTX-2.5 generation callback as `/run_studio`, launches a Gradio public URL, and registers that URL with Ella2027. Keep this Kaggle runtime alive while generating.

Required secrets: Kaggle Secret `ELLA_KAGGLE_WORKER_TOKEN`; Render API environment variable `ELLA_KAGGLE_WORKER_TOKEN` must contain the same newly rotated value. Do not paste the token into this notebook or GitHub.

In [ ]:
import os, sys, json, time, subprocess, shutil, requests
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import gradio as gr

WGP_DIR = Path('/kaggle/working/WanGP')
OUTPUT_DIR = Path('/kaggle/working/ella-gradio-output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_TYPE = 'ltx2_25_22B_distilled'

# Read the secret without ever printing it.
WORKER_TOKEN = UserSecretsClient().get_secret('ELLA_KAGGLE_WORKER_TOKEN').strip()
if not WORKER_TOKEN:
    raise RuntimeError('Missing Kaggle Secret: ELLA_KAGGLE_WORKER_TOKEN')

# Use the actual WanGP Python API from the existing model setup; initialize once.
os.chdir(WGP_DIR)
if str(WGP_DIR) not in sys.path:
    sys.path.insert(0, str(WGP_DIR))
from shared.api import init
print('[ELLA] Initializing WanGP API and loading model session...')
session = init(root=WGP_DIR, output_dir=OUTPUT_DIR, cli_args=['--attention', 'sdpa', '--profile', '4'], console_output=True, console_isatty=True)
defs = session.list_model_defs(model_type=MODEL_TYPE)
if not defs:
    raise RuntimeError(f'WanGP model definition not found: {MODEL_TYPE}')
availability = session.get_model_availability(MODEL_TYPE)
print('[ELLA] Model definition:', defs[0].get('name', MODEL_TYPE))
print('[ELLA] Model availability:', availability)
if isinstance(availability, dict) and str(availability.get('status', '')).lower() == 'missing':
    raise RuntimeError('LTX-2.5 model files/profile are not available. Re-run the model download/profile cell.')
print('[ELLA] Model session initialized.')

## 6. Real generation function and Gradio API

`run_studio` uses the in-memory WanGP session. The current app adapter requests one scene at a time. This profile supports up to 10 seconds per generated scene; it returns an MP4 and a status message. The current LTX integration does not create SRT subtitles or custom voice tracks.

In [ ]:
def _find_video(result):
    candidates = [Path(str(x)) for x in (getattr(result, 'generated_files', None) or [])]
    candidates += [Path(str(getattr(x, 'path', ''))) for x in (getattr(result, 'artifacts', None) or []) if getattr(x, 'media_type', None) == 'video' and getattr(x, 'path', None)]
    for p in reversed(candidates):
        if p.is_file() and p.stat().st_size and p.suffix.lower() in ('.mp4', '.mov', '.webm', '.mkv'):
            return p
    raise RuntimeError('WanGP finished but returned no saved video file. Check the WanGP output log.')


def run_studio(mode, scenes_mode, prompt, manual_scenes, narration, neg_prompt,
               ref_img, audio_in, enable_voice, voice, duration, num_scenes,
               base_res, out_w, out_h, fps, steps, guidance, seed, lip_sync, burn_sub):
    started = time.time()
    prompt = str(prompt or '').strip()
    if not prompt:
        return None, None, 'Error: prompt is empty.'
    seconds = int(duration or 5)
    if seconds < 1 or seconds > 10:
        return None, None, 'This Kaggle LTX-2.5 profile supports 1–10 seconds per scene.'
    try:
        defaults = session.get_default_settings(MODEL_TYPE)
        if not isinstance(defaults, dict) or not defaults:
            raise RuntimeError('WanGP returned no default model settings.')
        settings = dict(defaults)
        width, height = int(out_w or 832), int(out_h or 480)
        # Align dimensions to multiples of 32 for the LTX pipeline.
        width = max(320, (width // 32) * 32)
        height = max(320, (height // 32) * 32)
        settings.update({
            'model_type': MODEL_TYPE,
            'prompt': prompt,
            'resolution': f'{width}x{height}',
            'video_length': f'{seconds}s',
            'duration_seconds': seconds,
            'force_fps': int(fps or 24),
            'audio_prompt_type': '',
        })
        if neg_prompt:
            settings['negative_prompt'] = str(neg_prompt)
        if mode and 'image' in str(mode).lower() and ref_img:
            image_path = Path(str(ref_img))
            if not image_path.is_file():
                raise RuntimeError('Reference image file was not found in the Kaggle session.')
            settings['image_start'] = str(image_path)
        print(f'[ELLA] Generate: {width}x{height}, {seconds}s, {int(fps or 24)} FPS')
        job = session.submit_task(settings)
        result = job.result(timeout=60 * 60)
        if not getattr(result, 'success', False):
            errors = getattr(result, 'errors', None) or []
            detail = '; '.join(str(getattr(e, 'message', e)) for e in errors)
            raise RuntimeError(detail or 'WanGP generation failed.')
        video = _find_video(result)
        probe = subprocess.run(['ffprobe','-v','error','-show_entries','stream=codec_type','-of','json',str(video)],capture_output=True,text=True,timeout=30)
        if probe.returncode != 0:
            raise RuntimeError('ffprobe could not validate the generated video: ' + probe.stderr[-300:])
        streams = {x.get('codec_type') for x in json.loads(probe.stdout).get('streams', [])}
        if 'video' not in streams:
            raise RuntimeError('Generated file has no video stream.')
        target = OUTPUT_DIR / f'ella_{int(time.time())}_{video.name}'
        if video.resolve() != target.resolve():
            shutil.copy2(video, target)
        elapsed = time.time() - started
        return str(target), None, f'Completed successfully in {elapsed/60:.1f} minutes. Video stream verified.'
    except Exception as exc:
        print('[ELLA] Generation error:', repr(exc))
        return None, None, f'Generation failed: {str(exc)[:700]}'


with gr.Blocks(title='ELLA Kaggle Studio') as demo:
    gr.Markdown('# ELLA Kaggle Studio\nLTX-2.5 video generation connected to Ella2027')
    mode = gr.Dropdown(['Text to Video', 'Image to Video'], value='Text to Video', label='Mode')
    scenes_mode = gr.Dropdown(['Automatic', 'Manual'], value='Automatic', label='Scenes Mode')
    prompt = gr.Textbox(label='Prompt', lines=4)
    manual_scenes = gr.Textbox(label='Manual Scenes', visible=False)
    narration = gr.Textbox(label='Narration', visible=False)
    neg_prompt = gr.Textbox(label='Negative Prompt')
    ref_img = gr.Image(type='filepath', label='Reference Image')
    audio_in = gr.Audio(type='filepath', label='Audio Input', visible=False)
    enable_voice = gr.Checkbox(label='Enable Voice', value=False, visible=False)
    voice = gr.Dropdown(['default'], value='default', label='Voice', visible=False)
    duration = gr.Slider(1, 10, value=5, step=1, label='Duration (seconds)')
    num_scenes = gr.Slider(1, 1, value=1, step=1, label='Number of Scenes')
    base_res = gr.Dropdown([480, 576], value=480, label='Base Resolution')
    out_w = gr.Dropdown([832, 1024, 640, 480], value=832, label='Output Width')
    out_h = gr.Dropdown([480, 576, 640, 832], value=480, label='Output Height')
    fps = gr.Slider(8, 24, value=24, step=1, label='FPS')
    steps = gr.Slider(4, 30, value=20, step=1, label='Steps')
    guidance = gr.Slider(1, 10, value=5, step=0.5, label='Guidance')
    seed = gr.Number(value=-1, label='Seed')
    lip_sync = gr.Checkbox(label='Lip Sync', value=False, visible=False)
    burn_sub = gr.Checkbox(label='Burn Subtitles', value=False, visible=False)
    generate = gr.Button('Generate', variant='primary')
    video_out = gr.Video(label='Generated Video')
    srt_out = gr.File(label='Subtitles (not generated by this model)')
    status = gr.Textbox(label='Status')
    generate.click(fn=run_studio, inputs=[mode, scenes_mode, prompt, manual_scenes, narration, neg_prompt, ref_img, audio_in, enable_voice, voice, duration, num_scenes, base_res, out_w, out_h, fps, steps, guidance, seed, lip_sync, burn_sub], outputs=[video_out, srt_out, status], api_name='run_studio')

print('[ELLA] Launching Gradio share URL...')
local_url, public_url, share_error = demo.launch(server_name='0.0.0.0', server_port=7860, share=True, prevent_thread_lock=True, show_error=True)
if not public_url:
    raise RuntimeError(f'Gradio did not create a public share URL: {share_error}')
print('[ELLA] Public Gradio URL:', public_url)
bridge = requests.post('https://ella2027.onrender.com/set-kaggle-url', headers={'X-Worker-Token': WORKER_TOKEN}, json={'url': public_url}, timeout=30)
print('[ELLA] Render URL registration:', bridge.status_code, bridge.text[:500])
bridge.raise_for_status()
if not bridge.json().get('ok'):
    raise RuntimeError('Render did not accept the Gradio URL. Check ELLA_KAGGLE_WORKER_TOKEN on Render and Kaggle.')
print('\nREADY: open Ella2027 → Settings → Providers → Video → ELLA Kaggle GPU → Test. Keep this Kaggle session alive.')

## 7. Use Ella2027

1. Wait until the last cell prints `READY` and `Render URL registration: 200`.
2. In Ella2027 open **Settings → Providers → Video**, add/select **ELLA Kaggle GPU**, and paste the same worker token as the API key.
3. Press **Test**. It should confirm `/run_studio` is available.
4. Generate a short test scene (5 seconds first). Keep the Kaggle runtime active until the app finishes downloading and storing the MP4.

The notebook does not modify existing GitHub files. Kaggle GPU availability, session quotas, and first-run model downloads still apply. The model's current integration returns video only; custom voice, SRT subtitles, and lip-sync are not implemented here.